# EcoSentinel AI — Module B: Water Quality Training & Model Evaluation
**Academic ML Project:** Intelligent Waste and Water Monitoring System  
**Module:** Water Intelligence (CPCB Classifier & WQI Engine)  
**Model:** Random Forest Classifier (Class-Balanced Ensemble)  

### Objectives:
1. Preprocess raw data with zero data leakage (imputers and scalers fitted strictly on train split).
2. Map statutory CPCB criteria (Class A, Class B, Class C, Class D/E, Critical).
3. Benchmark comparative candidate models (Random Forest, Extra Trees, HistGradientBoosting) on validation set.
4. Train champion Random Forest with class-weighted cost-sensitive learning.
5. Evaluate performance on held-out test split: Accuracy, Macro/Weighted F1, Confusion Matrix, and Gini Feature Importances.
6. Save production model checkpoint to `models/water_quality_model.joblib`.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append('..')
from src.water.preprocess import preprocess_water_dataset
from src.water.train import train_water_model

print("Environment initialized.")

## 1. Execute Leakage-Free Preprocessing Pipeline
We partition the dataset into Train (70%), Val (15%), and Test (15%) splits, fit the RobustScaler pipeline strictly on the training partition, and export the preprocessor.

In [ ]:
prep_results = preprocess_water_dataset(
    raw_csv_path='../data/raw/water/cpcb_water_quality.csv',
    output_dir='../data/processed/water',
    model_dir='../models'
)
print(f"Train observations: {len(prep_results['train_df'])}")
print(f"Val observations:   {len(prep_results['val_df'])}")
print(f"Test observations:  {len(prep_results['test_df'])}")

## 2. Train and Evaluate Champion Model
We invoke `train_water_model()` to compare models on the validation split and evaluate the champion on the unseen test split.

In [ ]:
metrics = train_water_model(
    processed_dir='../data/processed/water',
    model_output_dir='../models',
    eval_output_dir='../outputs/evaluation'
)
print(f"Test Accuracy: {metrics['test_metrics']['accuracy']:.2%}")
print(f"Macro F1-Score: {metrics['test_metrics']['macro_f1']:.4f}")

## 3. Evaluation Artifacts & Diagnostic Dashboards
### A. Confusion Matrix (Raw Counts & Normalized Recall)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(Image.open('../outputs/evaluation/water_confusion_matrix.png'))
ax1.axis('off')
ax1.set_title("Test Confusion Matrix", fontsize=12)

ax2.imshow(Image.open('../outputs/evaluation/water_confusion_matrix_normalized.png'))
ax2.axis('off')
ax2.set_title("Normalized Confusion Matrix (Recall)", fontsize=12)
plt.tight_layout()
plt.show()

### B. Environmental Drivers (Feature Importance Rankings)

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(Image.open('../outputs/evaluation/water_feature_importance.png'))
plt.axis('off')
plt.show()

### C. Architecture Benchmark & Per-Class Performance Summary

In [ ]:
plt.figure(figsize=(14, 6))
plt.imshow(Image.open('../outputs/evaluation/water_results.png'))
plt.axis('off')
plt.show()